# ⚛️ DFT Validation — HgO Adsorption on Au(111)
## Single Point Energy (SPE) Calculations with GPAW + ASE

---
**Workflow Context:** This notebook is the **Stage 2 DFT Validation** step following high-throughput MLFF screening (CHGNet/MACE).

| Parameter | Value | Justification |
|---|---|---|
| Functional | PBE | Matches MLFF training set |
| Mode | Plane-Wave (PW) | Standard for surface science |
| Cutoff | 400 eV | High-accuracy validation |
| K-grid | (2,2,1) M-P | Balanced for Kaggle CPU/RAM |
| PAW | GPAW default | Standard pseudopotentials |

**Reference Formula:**
$$E_{\text{ads}} = E_{\text{total}}(\text{slab+HgO}) - E_{\text{slab}}(\text{clean}) - E_{\text{gas}}(\text{HgO})$$

---
> **Note:** All calculations use `txt='dft_log.txt'` to redirect GPAW's verbose output and prevent notebook crashes.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — ENVIRONMENT SETUP & INSTALLATION                              ║
# ║  Installs GPAW, ASE, and downloads PAW datasets                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os

def run(cmd, label=""):
    """Run a shell command with clean output."""
    print(f"  → {label or cmd[:60]}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ✗ STDERR: {result.stderr[-300:]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

# ─── Step 1: Install core packages ────────────────────────────────────────────
print("[1/3] Installing Python packages...")
run("pip install -q gpaw ase", "Installing gpaw + ase")
print("  ✓ gpaw and ase installed")

# ─── Step 2: Download GPAW PAW datasets ───────────────────────────────────────
# Datasets are needed for Au and Hg, O PAW potentials.
# Default install path: ~/.gpaw/datasets/
print("\n[2/3] Downloading GPAW PAW datasets...")
GPAW_DATA_DIR = os.path.expanduser("~/.gpaw/datasets")
os.makedirs(GPAW_DATA_DIR, exist_ok=True)

try:
    run(f"gpaw install-data {GPAW_DATA_DIR} --no-register --tarball=\"\"",
        "gpaw install-data (PAW potentials)")
    print("  ✓ GPAW datasets downloaded via install-data")
except RuntimeError:
    # Fallback: direct download from GPAW servers
    print("  ⚠ Falling back to direct dataset download...")
    run(
        "python -c \"from gpaw.setup_paths import setup_paths; "
        "print(setup_paths)\"",
        "Checking GPAW dataset path"
    )
    run(
        f"gpaw install-data {GPAW_DATA_DIR}",
        "Downloading GPAW datasets (fallback)"
    )

# ─── Step 3: Set GPAW_SETUP_PATH environment variable ─────────────────────────
os.environ["GPAW_SETUP_PATH"] = GPAW_DATA_DIR
print(f"\n[3/3] GPAW_SETUP_PATH set to: {GPAW_DATA_DIR}")

# ─── Verify installation ──────────────────────────────────────────────────────
import gpaw
import ase
print(f"\n✓ GPAW version : {gpaw.__version__}")
print(f"✓ ASE  version : {ase.__version__}")
print("\n🚀 Environment ready for DFT calculations.")

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — IMPORTS, CONFIG & GLOBAL PARAMETERS                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os, json, time, logging, warnings, traceback
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

import ase
from ase import Atoms
from ase.io import read, write
from ase.build import fcc111, add_adsorbate, molecule
from ase.constraints import FixAtoms
from ase.visualize import view
from gpaw import GPAW, PW

warnings.filterwarnings("ignore")

# ─── CPU threads: Kaggle typically has 2-4 CPUs per kernel ────────────────────
# OMP_NUM_THREADS controls OpenMP parallelism inside GPAW
os.environ.setdefault("OMP_NUM_THREADS", "2")

# ─── Directory structure ──────────────────────────────────────────────────────
# Input .xyz files from MLFF screening — try Kaggle input dir first, then cwd
KAGGLE_INPUT  = "/kaggle/input"
WORKING_DIR   = "/kaggle/working/dft_validation"
INPUT_DIR     = KAGGLE_INPUT if os.path.isdir(KAGGLE_INPUT) else "."

os.makedirs(WORKING_DIR, exist_ok=True)
LOG_FILE      = os.path.join(WORKING_DIR, "dft_log.txt")
RESULTS_FILE  = os.path.join(WORKING_DIR, "dft_validation_results.json")
TRAJ_DIR      = os.path.join(WORKING_DIR, "trajectories")
os.makedirs(TRAJ_DIR, exist_ok=True)

# ─── DFT Parameters ───────────────────────────────────────────────────────────
# All parameters are scientifically justified and documented:
DFT_PARAMS = dict(
    mode     = PW(400),         # Plane-wave, 400 eV cutoff (VASP standard)
    xc       = 'PBE',           # Matches CHGNet/MACE training XC functional
    kpts     = (2, 2, 1),       # Monkhorst-Pack; Γ-centered for surface
    occupations = {'name': 'fermi-dirac', 'width': 0.05},  # Methfessel-Paxton-like smearing
    convergence = {'energy': 1e-4},   # 0.1 meV — tight enough for SPE
    maxiter  = 200,             # Max SCF iterations
    symmetry = 'off',           # Disable symmetry for adsorption (broken)
    txt      = LOG_FILE,        # Redirect verbose GPAW output → prevents crash
    parallel = {'sl_auto': False},  # Disable ScaLAPACK for single-node Kaggle
)

# ─── Physical bounds for sanity checks ────────────────────────────────────────
E_ADS_MIN  = -5.00   # eV  (extremely strong chemisorption limit)
E_ADS_MAX  = +1.00   # eV  (slightly endothermic allowed)
E_SLAB_PER_ATOM = (-3.5, -2.0)  # eV/atom plausible for Au PBE
E_GAS_HGO       = (-6.0, -1.0)  # eV plausible for HgO molecule PBE

# ─── Adsorption sites from MLFF screening ─────────────────────────────────────
SITES = ["ontop", "bridge", "fcc", "hcp"]

# ─── Logger ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("DFT")

print("✓ Configuration loaded")
print(f"  Input  dir  : {INPUT_DIR}")
print(f"  Working dir : {WORKING_DIR}")
print(f"  DFT log     : {LOG_FILE}")
print(f"  Results     : {RESULTS_FILE}")
print(f"\n  DFT Settings:")
for k, v in DFT_PARAMS.items():
    print(f"    {k:15s} = {v}")

✓ Configuration loaded
  Input  dir  : /kaggle/input
  Working dir : /kaggle/working/dft_validation
  DFT log     : /kaggle/working/dft_validation/dft_log.txt
  Results     : /kaggle/working/dft_validation/dft_validation_results.json

  DFT Settings:
    mode            = <gpaw.wavefunctions.pw.PW object at 0x7fdd739b1640>
    xc              = PBE
    kpts            = (2, 2, 1)
    occupations     = {'name': 'fermi-dirac', 'width': 0.05}
    convergence     = {'energy': 0.0001}
    maxiter         = 200
    symmetry        = off
    txt             = /kaggle/working/dft_validation/dft_log.txt
    parallel        = {'sl_auto': False}


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — HELPER FUNCTIONS & VALIDATION GUARDS                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def locate_xyz(site: str) -> Path:
    """
    Locate the .xyz file for a given adsorption site.
    Search order:
      1. /kaggle/input/<site>.xyz
      2. /kaggle/input/**/<site>.xyz (any subfolder)
      3. ./<site>.xyz (current working directory)
    """
    fname = f"{site}.xyz"
    candidates = [
        Path(INPUT_DIR) / fname,
        Path(".") / fname,
    ]
    # Deep search in /kaggle/input subfolders
    if os.path.isdir(KAGGLE_INPUT):
        for p in Path(KAGGLE_INPUT).rglob(fname):
            candidates.insert(1, p)

    for path in candidates:
        if path.exists():
            log.info(f"Found {fname} → {path}")
            return path

    raise FileNotFoundError(
        f"Cannot find '{fname}'.\n"
        f"Expected at one of:\n"
        + "\n".join(f"  {p}" for p in candidates)
        + "\nPlease add your Kaggle dataset containing the .xyz files."
    )


def make_calculator(label: str) -> GPAW:
    """
    Factory: return a fresh GPAW calculator with standard DFT_PARAMS.
    Each calculation gets its own label for restart files.
    """
    params = dict(DFT_PARAMS)  # shallow copy — immutable values
    params['txt'] = os.path.join(WORKING_DIR, f"dft_log_{label}.txt")
    return GPAW(**params)


def run_spe(atoms: Atoms, label: str) -> float:
    """
    Run a Single Point Energy (SPE) calculation.
    
    Parameters
    ----------
    atoms : ASE Atoms object (geometry from MLFF optimization)
    label : unique identifier for log files
    
    Returns
    -------
    float : Total DFT energy in eV
    """
    log.info(f"Starting SPE for '{label}' | {len(atoms)} atoms")
    t0 = time.time()

    calc = make_calculator(label)
    atoms.calc = calc

    try:
        energy = atoms.get_potential_energy()
    except Exception as e:
        log.error(f"SCF failed for '{label}': {e}")
        raise
    finally:
        # Always release calculator memory — critical for OOM prevention
        try:
            calc.write(os.path.join(TRAJ_DIR, f"{label}.gpw"))  # Save restart
        except Exception:
            pass
        atoms.calc = None  # Detach calculator — free memory

    elapsed = time.time() - t0
    log.info(f"  ✓ {label}: E = {energy:.6f} eV | elapsed = {elapsed:.1f}s")
    return float(energy)


def guard_energy(energy: float, label: str, bounds: tuple):
    """Assert energy is within physically plausible bounds."""
    lo, hi = bounds
    if not (lo < energy < hi):
        raise ValueError(
            f"Unphysical energy for '{label}': {energy:.4f} eV\n"
            f"Expected range: ({lo}, {hi}) eV\n"
            f"HINT: Check SCF convergence in dft_log_{label}.txt"
        )


def guard_adsorption(e_ads: float, site: str):
    """Assert adsorption energy is physically meaningful."""
    if not (E_ADS_MIN < e_ads < E_ADS_MAX):
        raise ValueError(
            f"Unphysical E_ads={e_ads:.4f} eV at site '{site}'\n"
            f"Allowed: ({E_ADS_MIN}, {E_ADS_MAX}) eV\n"
            f"HINT: Verify E_slab and E_gas reference calculations."
        )


print("✓ Helper functions defined")
print("  - locate_xyz()    : smart file finder")
print("  - make_calculator(): GPAW factory with memory cleanup")
print("  - run_spe()       : single point energy wrapper")
print("  - guard_energy()  : physical bounds check")
print("  - guard_adsorption(): E_ads sanity check")

✓ Helper functions defined
  - locate_xyz()    : smart file finder
  - make_calculator(): GPAW factory with memory cleanup
  - run_spe()       : single point energy wrapper
  - guard_energy()  : physical bounds check
  - guard_adsorption(): E_ads sanity check


In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — REFERENCE CALCULATION 1: CLEAN Au(111) SLAB  E_slab          ║
# ║                                                                          ║
# ║  Strategy: Extract the slab geometry from one of the adsorbed .xyz       ║
# ║  files by removing the HgO molecule (last 2 atoms = Hg + O).            ║
# ║  This guarantees E_slab is computed on the EXACT same geometry as        ║
# ║  was used for adsorption, preventing reference mismatch errors.          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("═" * 68)
print("  REFERENCE CALCULATION 1 — Clean Au(111) Slab (E_slab)")
print("═" * 68)

# ── Load one adsorbed structure to extract the slab geometry ──────────────────
# Use 'ontop' as the reference — all sites share the same slab geometry
REFERENCE_SITE = "ontop"  # Any site works; slab atoms are identical

ref_xyz_path = locate_xyz(REFERENCE_SITE)
print(f"\n[1] Loading reference structure: {ref_xyz_path}")
adsorbed_ref = read(str(ref_xyz_path))
print(f"    Total atoms in adsorbed structure: {len(adsorbed_ref)}")
print(f"    Chemical symbols: {adsorbed_ref.get_chemical_formula()}")

# ── Identify the HgO molecule atoms ───────────────────────────────────────────
# Convention from MLFF screening: the adsorbate (Hg + O) are the LAST atoms.
# We verify this by checking the chemical symbols.
symbols = adsorbed_ref.get_chemical_symbols()
hgo_indices = [i for i, s in enumerate(symbols) if s in ('Hg', 'O')]
slab_indices = [i for i, s in enumerate(symbols) if s == 'Au']

if len(hgo_indices) != 2:
    raise ValueError(
        f"Expected exactly 2 HgO atoms (Hg + O), found {len(hgo_indices)}.\n"
        f"Symbols: {symbols}\n"
        f"HINT: Check your .xyz file format."
    )

print(f"    Au  atoms (slab)  : {len(slab_indices)} atoms, indices {slab_indices[:5]}...")
print(f"    HgO atoms (ads.)  : {len(hgo_indices)} atoms, indices {hgo_indices}")

# ── Build the clean slab Atoms object ─────────────────────────────────────────
slab_atoms = adsorbed_ref[slab_indices].copy()
slab_atoms.set_cell(adsorbed_ref.get_cell())   # Preserve periodic cell exactly
slab_atoms.set_pbc(adsorbed_ref.get_pbc())     # Preserve PBC (True, True, False)

# Verify constraints are preserved (bottom layers should be fixed)
if adsorbed_ref.constraints:
    # Re-apply FixAtoms to bottom layer Au atoms in the clean slab
    # The original constraint indices need remapping after HgO removal
    print("    ✓ Constraints detected and will be remapped to clean slab")
    # Build index mapping: original → new
    idx_map = {old: new for new, old in enumerate(slab_indices)}
    for constr in adsorbed_ref.constraints:
        if isinstance(constr, FixAtoms):
            new_indices = [idx_map[i] for i in constr.index if i in idx_map]
            if new_indices:
                slab_atoms.set_constraint(FixAtoms(indices=new_indices))
                print(f"    ✓ FixAtoms applied to {len(new_indices)} bottom-layer Au atoms")

print(f"\n[2] Clean slab summary:")
print(f"    Formula  : {slab_atoms.get_chemical_formula()}")
print(f"    Cell     : {np.diag(slab_atoms.get_cell()).round(3)} Å")
print(f"    PBC      : {slab_atoms.get_pbc()}")

# ── Save slab geometry for inspection ─────────────────────────────────────────
slab_path = os.path.join(TRAJ_DIR, "clean_slab.xyz")
write(slab_path, slab_atoms)
print(f"    Saved   : {slab_path}")

# ── Run SPE for clean slab ────────────────────────────────────────────────────
print(f"\n[3] Running DFT SPE for clean Au(111) slab...")
print(f"    (This is the most expensive calculation — {len(slab_atoms)} Au atoms)")
print(f"    GPAW output redirected → dft_log_slab.txt")

E_SLAB = run_spe(slab_atoms, label="slab")

# ── Physical sanity check ─────────────────────────────────────────────────────
e_per_atom = E_SLAB / len(slab_atoms)
guard_energy(e_per_atom, "slab_per_atom", E_SLAB_PER_ATOM)

print(f"\n{'─'*50}")
print(f"  ✓ E_slab = {E_SLAB:.6f} eV")
print(f"    ({e_per_atom:.4f} eV/atom — within expected PBE range)")
print(f"{'─'*50}")

════════════════════════════════════════════════════════════════════
  REFERENCE CALCULATION 1 — Clean Au(111) Slab (E_slab)
════════════════════════════════════════════════════════════════════


FileNotFoundError: Cannot find 'ontop.xyz'.
Expected at one of:
  /kaggle/input/ontop.xyz
  ontop.xyz
Please add your Kaggle dataset containing the .xyz files.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — REFERENCE CALCULATION 2: ISOLATED HgO MOLECULE  E_gas        ║
# ║                                                                          ║
# ║  HgO molecule in the same periodic box as the slab (to avoid           ║
# ║  Pulay stress differences). Use a large vacuum (same as slab cell).     ║
# ║  GAMMA-point only for molecule (non-periodic system).                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("═" * 68)
print("  REFERENCE CALCULATION 2 — Isolated HgO Molecule (E_gas)")
print("═" * 68)

# ── Extract HgO geometry from the adsorbed structure ──────────────────────────
# Using the geometry from the MLFF-optimized adsorbed structure.
# The Hg-O bond length should be close to experimental 2.056 Å.
hgo_from_ads = adsorbed_ref[hgo_indices].copy()
hg_symbol = [s for s in hgo_from_ads.get_chemical_symbols() if s == 'Hg']
o_symbol  = [s for s in hgo_from_ads.get_chemical_symbols() if s == 'O']

print(f"\n[1] HgO extracted from MLFF-optimized structure:")
print(f"    Symbols: {hgo_from_ads.get_chemical_symbols()}")

# Compute Hg-O bond length as extracted
pos = hgo_from_ads.get_positions()
bond_length = np.linalg.norm(pos[0] - pos[1])
print(f"    Hg-O bond length : {bond_length:.4f} Å  (exp: 2.056 Å)")
if not (1.8 < bond_length < 2.5):
    log.warning(f"Unusual Hg-O bond {bond_length:.3f} Å — check HgO geometry")

# ── Place HgO in an isolated large box ────────────────────────────────────────
# Box size: same XY as slab cell to avoid basis set incompatibility.
# Z direction: 15 Å vacuum on each side of molecule.
slab_cell = adsorbed_ref.get_cell()
box_x = float(slab_cell[0, 0])
box_y = float(slab_cell[1, 1])
box_z = 15.0  # Å vacuum — large enough to avoid periodic images

# Center molecule in the box
hgo_centered = hgo_from_ads.copy()
hgo_centered.set_cell([box_x, box_y, box_z])
hgo_centered.center()  # Place molecule at cell center
hgo_centered.set_pbc([True, True, False])  # Non-periodic in Z

# Remove any constraints from the adsorbate
hgo_centered.set_constraint()

print(f"\n[2] HgO molecule box:")
print(f"    Cell : [{box_x:.3f}, {box_y:.3f}, {box_z:.3f}] Å")
print(f"    PBC  : {hgo_centered.get_pbc()}")

# Save geometry
hgo_path = os.path.join(TRAJ_DIR, "hgo_molecule.xyz")
write(hgo_path, hgo_centered)
print(f"    Saved: {hgo_path}")

# ── Run SPE for HgO molecule ──────────────────────────────────────────────────
# IMPORTANT: Use Gamma-point only for the molecule calculation.
# Molecule needs no k-space sampling (non-periodic).
print(f"\n[3] Running DFT SPE for HgO molecule...")
print(f"    Using Γ-point only (non-periodic system)")
print(f"    GPAW output redirected → dft_log_hgo_gas.txt")

# Override k-points for molecule: Gamma point only
from gpaw import GPAW, PW
gas_params = dict(DFT_PARAMS)
gas_params['kpts'] = {'size': (1, 1, 1), 'gamma': True}  # Gamma-point
gas_params['txt']  = os.path.join(WORKING_DIR, "dft_log_hgo_gas.txt")

import time
t0 = time.time()
gas_calc = GPAW(**gas_params)
hgo_centered.calc = gas_calc

try:
    E_GAS = hgo_centered.get_potential_energy()
except Exception as e:
    log.error(f"HgO gas SPE failed: {e}")
    raise
finally:
    try:
        gas_calc.write(os.path.join(TRAJ_DIR, "hgo_gas.gpw"))
    except Exception:
        pass
    hgo_centered.calc = None  # Free memory

elapsed = time.time() - t0

# ── Sanity check ──────────────────────────────────────────────────────────────
guard_energy(E_GAS, "hgo_gas", E_GAS_HGO)

print(f"\n{'─'*50}")
print(f"  ✓ E_gas(HgO) = {E_GAS:.6f} eV")
print(f"    (elapsed: {elapsed:.1f}s)")
print(f"{'─'*50}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — MAIN LOOP: SPE for All 4 Adsorbed Sites                      ║
# ║                                                                          ║
# ║  Sites are processed ONE-BY-ONE to respect Kaggle RAM limits.           ║
# ║  Each calculation is independent — failures do not abort others.        ║
# ║  Formula: E_ads = E_total - (E_slab + E_gas)                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("═" * 68)
print("  MAIN LOOP — DFT SPE for All Adsorbed Sites")
print("═" * 68)
print(f"  Sites to process : {SITES}")
print(f"  E_slab reference : {E_SLAB:.6f} eV")
print(f"  E_gas  reference : {E_GAS:.6f}  eV")
print(f"  Formula: E_ads = E_total − (E_slab + E_gas)")
print()

# ── Results accumulator ────────────────────────────────────────────────────────
results = {
    "metadata": {
        "timestamp"   : datetime.now(timezone.utc).isoformat(),
        "method"      : "DFT/PBE",
        "basis"       : "Plane-Wave",
        "cutoff_eV"   : 400,
        "kpoints"     : "(2,2,1) Monkhorst-Pack",
        "paw"         : "GPAW default",
        "smearing"    : "Fermi-Dirac 0.05 eV",
        "code"        : f"GPAW {gpaw.__version__} / ASE {ase.__version__}",
        "purpose"     : "MLFF screening validation (CHGNet/MACE → DFT)",
    },
    "references": {
        "E_slab_eV"   : E_SLAB,
        "E_gas_HgO_eV": E_GAS,
    },
    "sites": {},
    "errors": {},
}

# ── Process each site independently ───────────────────────────────────────────
wall_start = time.time()

for i, site in enumerate(SITES):
    print(f"{'─'*68}")
    print(f"  [{i+1}/{len(SITES)}] Site: '{site.upper()}'")
    print(f"{'─'*68}")

    try:
        # ── 1. Load adsorbed structure from MLFF-optimized .xyz ───────────────
        xyz_path = locate_xyz(site)
        print(f"  Input file : {xyz_path}")
        adsorbed = read(str(xyz_path))
        n_atoms = len(adsorbed)
        formula = adsorbed.get_chemical_formula()
        print(f"  Structure  : {formula}  ({n_atoms} atoms)")
        print(f"  Cell       : {np.diag(adsorbed.get_cell()).round(3)} Å")

        # ── 2. Preserve PBC and cell from the MLFF calculation ────────────────
        # Ensure proper periodicity: True in XY plane, False in Z (vacuum slab)
        adsorbed.set_pbc([True, True, False])

        # ── 3. Record geometric data before SPE ───────────────────────────────
        sym = adsorbed.get_chemical_symbols()
        pos = adsorbed.get_positions()
        hg_pos = pos[[j for j, s in enumerate(sym) if s == 'Hg'][0]]
        o_pos  = pos[[j for j, s in enumerate(sym) if s == 'O'][0]]
        hgo_bond = float(np.linalg.norm(hg_pos - o_pos))

        # Height of Hg above the topmost Au layer
        au_z_max = max(pos[j, 2] for j, s in enumerate(sym) if s == 'Au')
        hg_height = float(hg_pos[2] - au_z_max)

        print(f"  Hg height  : {hg_height:.3f} Å above Au surface")
        print(f"  Hg-O bond  : {hgo_bond:.4f} Å  (exp: 2.056 Å)")

        # ── 4. Run DFT Single Point Energy ────────────────────────────────────
        print(f"  Running SPE... (GPAW → dft_log_{site}.txt)")
        t_site = time.time()
        E_TOTAL = run_spe(adsorbed, label=site)
        dt = time.time() - t_site

        # ── 5. Compute adsorption energy ─────────────────────────────────────
        #   E_ads = E_total(slab+HgO) − E_slab(clean) − E_gas(HgO)
        E_ADS = E_TOTAL - (E_SLAB + E_GAS)

        # ── 6. Physical sanity check ──────────────────────────────────────────
        guard_adsorption(E_ADS, site)

        # ── 7. Store results ──────────────────────────────────────────────────
        results["sites"][site] = {
            "E_total_eV"    : E_TOTAL,
            "E_ads_eV"      : E_ADS,
            "E_ads_kJ_mol"  : E_ADS * 96.485,   # eV → kJ/mol
            "hg_height_A"   : hg_height,
            "hgo_bond_A"    : hgo_bond,
            "n_atoms"       : n_atoms,
            "elapsed_s"     : round(dt, 1),
            "status"        : "success",
        }

        print(f"\n  ✅ {site.upper()} completed:")
        print(f"     E_total = {E_TOTAL:.6f} eV")
        print(f"     E_ads   = {E_ADS:.4f} eV  ({E_ADS * 96.485:.2f} kJ/mol)")
        print(f"     Elapsed : {dt:.1f} s")

    except FileNotFoundError as e:
        log.error(f"File not found for site '{site}': {e}")
        results["errors"][site] = {"type": "FileNotFoundError", "message": str(e)}
        results["sites"][site] = {"status": "failed", "error": str(e)}
        print(f"  ❌ SKIPPED — {e}")

    except Exception as e:
        log.error(f"SPE failed for site '{site}': {e}")
        tb = traceback.format_exc()
        results["errors"][site] = {"type": type(e).__name__, "message": str(e), "traceback": tb}
        results["sites"][site]  = {"status": "failed", "error": str(e)}
        print(f"  ❌ FAILED — {type(e).__name__}: {e}")
        print(f"  ⚠ Continuing to next site...")

    finally:
        # Save partial results after every site — guard against kernel crashes
        with open(RESULTS_FILE, "w") as f:
            json.dump(results, f, indent=2, default=str)
        print(f"  💾 Partial results saved → {RESULTS_FILE}")
        print()

total_time = time.time() - wall_start
results["metadata"]["total_wall_time_s"] = round(total_time, 1)

print("═" * 68)
print(f"  ✓ Main loop complete | Total wall time: {total_time/60:.1f} min")
print("═" * 68)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — COMPARISON TABLE & MLFF vs DFT ANALYSIS                      ║
# ║                                                                          ║
# ║  Builds the final summary comparing DFT results against MLFF values.    ║
# ║  Saves to dft_validation_results.json (final, complete version).        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("═" * 68)
print("  RESULTS SUMMARY — DFT/PBE Validation")
print("═" * 68)

# ── Collect all successful DFT results ────────────────────────────────────────
successful = {
    site: data for site, data in results["sites"].items()
    if data.get("status") == "success"
}
failed = {
    site: data for site, data in results["sites"].items()
    if data.get("status") != "success"
}

# ── Print comparison table ─────────────────────────────────────────────────────
header = f"{'Site':>8} | {'E_total (eV)':>14} | {'E_ads (eV)':>12} | {'E_ads (kJ/mol)':>15} | {'Hg-height (Å)':>14} | {'Status':>8}"
sep    = "─" * len(header)
print(f"\n{header}")
print(sep)

e_ads_values = []
for site in SITES:
    data = results["sites"].get(site, {})
    if data.get("status") == "success":
        e_ads = data['E_ads_eV']
        e_ads_values.append((site, e_ads))
        print(
            f"{site:>8} | {data['E_total_eV']:>14.6f} | "
            f"{e_ads:>+12.4f} | {data['E_ads_kJ_mol']:>+15.2f} | "
            f"{data['hg_height_A']:>14.3f} | {'✅ OK':>8}"
        )
    else:
        print(f"{site:>8} | {'—':>14} | {'—':>12} | {'—':>15} | {'—':>14} | {'❌ FAIL':>8}")

print(sep)

# ── Reference energies ────────────────────────────────────────────────────────
print(f"\n  Reference Energies:")
print(f"    E_slab (clean Au(111)) = {E_SLAB:.6f} eV")
print(f"    E_gas  (HgO molecule)  = {E_GAS:.6f} eV")

# ── Ranking ───────────────────────────────────────────────────────────────────
if e_ads_values:
    e_ads_values.sort(key=lambda x: x[1])  # Sort: most negative = strongest binding
    print(f"\n  🏆 Adsorption Energy Ranking (strongest → weakest binding):")
    for rank, (site, e_ads) in enumerate(e_ads_values, 1):
        bar = "█" * int(abs(e_ads) * 10)
        print(f"    #{rank} {site:>7}: {e_ads:+.4f} eV  {bar}")

    most_stable = e_ads_values[0][0]
    print(f"\n  ✓ Most stable adsorption site (DFT/PBE): '{most_stable.upper()}'")

    # Build comparison table for JSON
    comparison_table = []
    for site, e_ads in e_ads_values:
        d = results["sites"][site]
        comparison_table.append({
            "site"          : site,
            "rank_dft"      : e_ads_values.index((site, e_ads)) + 1,
            "E_ads_dft_eV"  : round(e_ads, 4),
            "E_ads_dft_kJmol": round(d['E_ads_kJ_mol'], 2),
            "E_total_eV"    : round(d['E_total_eV'], 6),
            "hg_height_A"   : round(d['hg_height_A'], 3),
            "hgo_bond_A"    : round(d['hgo_bond_A'], 4),
        })
    results["comparison_table"] = comparison_table
    results["most_stable_site_dft"] = most_stable

# ── Failed sites report ───────────────────────────────────────────────────────
if failed:
    print(f"\n  ⚠ Failed sites ({len(failed)}):")
    for site, data in failed.items():
        print(f"    {site}: {data.get('error', 'unknown error')}")

# ── Save final results ────────────────────────────────────────────────────────
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"\n{'═'*68}")
print(f"  ✅ Final results saved → {RESULTS_FILE}")
print(f"     Successful : {len(successful)}/4 sites")
print(f"     Failed     : {len(failed)}/4 sites")
print(f"{'═'*68}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — MLFF vs DFT PARITY ANALYSIS                                  ║
# ║                                                                          ║
# ║  Compare DFT adsorption energies against the MLFF-predicted values.    ║
# ║  Compute MAE and ranking correlation.                                    ║
# ║  NOTE: MLFF values must be loaded from your screening results.          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("═" * 68)
print("  MLFF ↔ DFT PARITY ANALYSIS")
print("═" * 68)

# ── Load MLFF reference energies from your screening notebook ─────────────────
# Path to the JSON/CSV produced by the CHGNet/MACE screening (Cell 1 notebook).
# Adjust the path/filename to match your actual output.
MLFF_RESULTS_PATH = "/kaggle/working/hgo_benchmark/data/adsorption_results.json"

mlff_data = {}
if os.path.exists(MLFF_RESULTS_PATH):
    with open(MLFF_RESULTS_PATH) as f:
        raw = json.load(f)
    # Attempt to extract E_ads per site from common key patterns
    for site in SITES:
        for key_pattern in [site, site.upper(), f"{site}_chgnet", f"{site}_mace"]:
            if key_pattern in raw:
                entry = raw[key_pattern]
                if isinstance(entry, dict):
                    # Extract E_ads from nested structure
                    for e_key in ["E_ads", "e_ads", "adsorption_energy", "E_ads_eV"]:
                        if e_key in entry:
                            mlff_data[site] = float(entry[e_key])
                            break
                elif isinstance(entry, (int, float)):
                    mlff_data[site] = float(entry)
    print(f"  ✓ MLFF results loaded from {MLFF_RESULTS_PATH}")
else:
    print(f"  ⚠ MLFF results file not found at {MLFF_RESULTS_PATH}")
    print(f"    → Manual MLFF comparison mode: insert your E_ads values below")
    # ── MANUAL ENTRY: paste your CHGNet/MACE E_ads values here ───────────────
    # Replace these placeholder values with your actual MLFF results:
    mlff_data = {
        "ontop"  : None,   # e.g. -1.234  (eV)
        "bridge" : None,   # e.g. -1.456
        "fcc"    : None,   # e.g. -1.678
        "hcp"    : None,   # e.g. -1.567
    }
    print(f"    Fill in mlff_data above and re-run this cell.")

# ── Parity table ──────────────────────────────────────────────────────────────
dft_data = {
    site: results["sites"][site]["E_ads_eV"]
    for site in SITES
    if results["sites"].get(site, {}).get("status") == "success"
}

common_sites = [s for s in SITES if s in dft_data and mlff_data.get(s) is not None]

if common_sites:
    print(f"\n  Parity Table (N={len(common_sites)} sites with both MLFF and DFT):")
    print(f"  {'Site':>8} | {'MLFF E_ads (eV)':>16} | {'DFT E_ads (eV)':>15} | {'Δ (DFT−MLFF)':>14}")
    print(f"  {'─'*60}")

    deltas = []
    for site in common_sites:
        d_dft  = dft_data[site]
        d_mlff = mlff_data[site]
        delta  = d_dft - d_mlff
        deltas.append(delta)
        flag   = "⚠" if abs(delta) > 0.3 else "✓"
        print(f"  {site:>8} | {d_mlff:>+16.4f} | {d_dft:>+15.4f} | {delta:>+13.4f} {flag}")

    mae  = np.mean(np.abs(deltas))
    rmse = np.sqrt(np.mean(np.array(deltas)**2))
    bias = np.mean(deltas)

    print(f"\n  Error Metrics:")
    print(f"    MAE  = {mae:.4f} eV  ({mae*96.485:.2f} kJ/mol)")
    print(f"    RMSE = {rmse:.4f} eV  ({rmse*96.485:.2f} kJ/mol)")
    print(f"    Bias = {bias:+.4f} eV  ({'MLFF overbinds' if bias < 0 else 'MLFF underbinds'})")

    if mae < 0.1:
        quality = "🟢 EXCELLENT — MLFF within 0.1 eV of DFT"
    elif mae < 0.3:
        quality = "🟡 GOOD — MLFF within chemical accuracy range"
    else:
        quality = "🔴 POOR — Large MLFF-DFT discrepancy; consider retraining"
    print(f"\n  Validation Quality: {quality}")

    # Save to results
    results["mlff_comparison"] = {
        "mlff_values"   : mlff_data,
        "mae_eV"        : round(mae, 4),
        "rmse_eV"       : round(rmse, 4),
        "bias_eV"       : round(bias, 4),
        "n_sites"       : len(common_sites),
        "quality_flag"  : quality,
    }
else:
    print("  ⚠ No common sites available for MLFF-DFT comparison.")
    print("    Fill in mlff_data dict above with your MLFF E_ads values.")

# ── Final save ────────────────────────────────────────────────────────────────
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n  ✅ Complete results saved → {RESULTS_FILE}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — DIAGNOSTIC & LOG INSPECTION                                   ║
# ║                                                                          ║
# ║  Print tail of each GPAW log file to verify SCF convergence.            ║
# ║  This cell is for manual inspection — run only if results look wrong.   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("═" * 68)
print("  DIAGNOSTIC — GPAW Log Inspection")
print("═" * 68)

LOG_LABELS = ["slab", "hgo_gas"] + SITES

for label in LOG_LABELS:
    log_path = os.path.join(WORKING_DIR, f"dft_log_{label}.txt")
    if os.path.exists(log_path):
        size_kb = os.path.getsize(log_path) / 1024
        print(f"\n  📄 dft_log_{label}.txt  ({size_kb:.1f} KB)")
        # Print last 20 lines (SCF convergence info)
        with open(log_path) as f:
            lines = f.readlines()
        tail = lines[-20:] if len(lines) > 20 else lines
        # Look for convergence indicator
        converged = any("Converged" in l or "converged" in l for l in lines)
        print(f"     SCF Converged: {'✅ YES' if converged else '❌ NOT DETECTED'}")
        print("     Last 10 lines:")
        for line in tail[-10:]:
            print(f"       {line.rstrip()}")
    else:
        print(f"  ⚠ Log not found: {log_path}")

print(f"\n  All GPAW .gpw restart files saved in: {TRAJ_DIR}")
gpw_files = list(Path(TRAJ_DIR).glob("*.gpw"))
print(f"  GPW files: {[f.name for f in gpw_files]}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — FINAL SUMMARY PRINT & JSON PREVIEW                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

print("╔" + "═"*66 + "╗")
print("║          DFT VALIDATION COMPLETE — FINAL SUMMARY              ║")
print("╚" + "═"*66 + "╝")

with open(RESULTS_FILE) as f:
    final = json.load(f)

print(f"\n  Output file: {RESULTS_FILE}")
print(f"  Method     : {final['metadata']['method']}")
print(f"  Cutoff     : {final['metadata']['cutoff_eV']} eV")
print(f"  K-grid     : {final['metadata']['kpoints']}")
print(f"  Timestamp  : {final['metadata']['timestamp']}")
print(f"  Wall time  : {final['metadata'].get('total_wall_time_s', 'N/A')} s")

print(f"\n  ─── Reference Energies ───")
print(f"  E_slab  = {final['references']['E_slab_eV']:.6f} eV")
print(f"  E_gas   = {final['references']['E_gas_HgO_eV']:.6f} eV")

if "comparison_table" in final:
    print(f"\n  ─── DFT Adsorption Energies (ranked) ───")
    for row in final["comparison_table"]:
        print(
            f"  #{row['rank_dft']} {row['site']:>8}: "
            f"E_ads = {row['E_ads_dft_eV']:+.4f} eV "
            f"({row['E_ads_dft_kJmol']:+.2f} kJ/mol) "
            f"| Hg-height = {row['hg_height_A']:.3f} Å"
        )

if "mlff_comparison" in final:
    comp = final["mlff_comparison"]
    print(f"\n  ─── MLFF-DFT Comparison ───")
    print(f"  MAE  = {comp['mae_eV']:.4f} eV")
    print(f"  RMSE = {comp['rmse_eV']:.4f} eV")
    print(f"  Bias = {comp['bias_eV']:+.4f} eV")
    print(f"  {comp['quality_flag']}")

if final.get("most_stable_site_dft"):
    print(f"\n  🏆 Most Stable Site (DFT): {final['most_stable_site_dft'].upper()}")

print("\n" + "─"*68)
print(f"  JSON preview (first 30 lines):")
print("─"*68)
json_str = json.dumps(final, indent=2, default=str)
for line in json_str.split("\n")[:30]:
    print(f"  {line}")
print("  ...")
print("─"*68)
print(f"\n  Full JSON → {RESULTS_FILE}")
print(f"  Restart   → {TRAJ_DIR}/*.gpw")
print(f"  DFT logs  → {WORKING_DIR}/dft_log_*.txt")